In [ ]:
# pull to colab from git

!git clone https://github.com/Valerik-Bogd/restore_punct.git

# 2. Enter the folder
%cd restore_punct

# 3. Pull latest changes (run this if you updated code locally)
!git pull

In [ ]:
# adapt to colab gpu / nvidia 5060laptop setup

import os
import sys
import torch
import warnings

MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"
MAX_LEN = 512
BATCH_SIZE = 8 #####
DEBUG = True

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")

    # Mount Drive
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install -q transformers datasets seqeval evaluate accelerate -q

    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/"
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    GRAD_ACCUM = 1

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    # RTX 5060 (8GB VRAM)
    BATCH_SIZE = 4
    NUM_WORKERS = 4
    GRAD_ACCUM = 2

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
import torch
import json
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, load_dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
import torch.nn as nn
import pandas as pd


In [ ]:
print(torch.cuda.is_available())
# !nvidia-smi


True
Mon Dec 29 16:52:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning, message=".*`tokenizer` is deprecated.*")

warnings.filterwarnings("ignore", message=".*seems not to be NE tag.*")


warnings.filterwarnings("ignore", message=".*UndefinedMetricWarning.*")


In [ ]:
data_files = {
    "train": os.path.join(DATA_DIR, "train_all.json"),
    "validation": os.path.join(DATA_DIR, "val_internal.json")
}

raw_datasets = load_dataset("json", data_files=data_files)

print(raw_datasets)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 19087
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1005
    })
})


In [ ]:
with open(os.path.join(DATA_DIR, "label_map.json"), 'r', encoding='utf-8') as f:
    ID_TO_LABEL_RAW = json.load(f)
    id2label = {int(k): v for k, v in ID_TO_LABEL_RAW.items()}
    label2id = {v: k for k, v in id2label.items()}

LABELS = list(id2label.values())
NUM_LABELS = len(LABELS)

print(f"Loaded {NUM_LABELS} labels.")
# print(LABELS)

Loaded 28 labels.


In [ ]:
def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets label
            else:
                label_ids.append(-100) # Subsequent subwords ignored
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
"""
looks like this:
{
  "tokens": ["Hello", ",", "world", "!"],
  "ner_tags": [0, 1, 0, 2]
}
"""

'\nlooks like this:\n{\n  "tokens": ["Hello", ",", "world", "!"],\n  "ner_tags": [0, 1, 0, 2]\n}\n'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


tokenized_dataset = raw_datasets.map(align_labels_with_tokens, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/19087 [00:00<?, ? examples/s]

Map:   0%|          | 0/1005 [00:00<?, ? examples/s]

In [ ]:
if DEBUG:
    print("DEBUGGING")
    # Take only first 50 samples for training and 10 for validation
    raw_datasets["train"] = raw_datasets["train"].select(range(50))
    raw_datasets["validation"] = raw_datasets["validation"].select(range(10))

    NUM_EPOCHS = 1
    logging_steps = 5

    print(f"Train size: {len(raw_datasets['train'])}")
    print(f"Val size:   {len(raw_datasets['validation'])}")
else:
    print("Debugging = False")
    NUM_EPOCHS = 3
    logging_steps = 100


DEBUGGING
Train size: 50
Val size:   10


In [ ]:
def get_baseline_model():
    return AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )


In [ ]:
# LSTM
class BertLSTM(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.config = self.bert.config

        # LSTM Layer
        self.lstm = nn.LSTM(input_size=768, hidden_size=256, batch_first=True, bidirectional=True)

        # Classifier
        self.classifier = nn.Linear(256 * 2, num_labels) # Bidirectional
        self.num_labels = num_labels
        self.loss_fct = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = outputs.last_hidden_state

        lstm_output, _ = self.lstm(sequence_output)

        # Linear Layer
        logits = self.classifier(lstm_output)

        loss = None
        if labels is not None:
            loss = self.loss_fct(logits.view(-1, self.num_labels), labels.view(-1)) # Flatten to calculate loss

        return (loss, logits) if loss is not None else logits

In [ ]:
seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # list comprehension - flatten and filter -100 tokens
    true_predictions = [
        LABELS[p]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    true_labels = [
        LABELS[l]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    # 2. Compute overall metrics
    # weighted: takes class imbalance into account (useful since 'O' is very frequent)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average='weighted', zero_division=0
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    # F1 score for specifically "COMMA" vs "PERIOD" etc.
    report = classification_report(true_labels, true_predictions, zero_division=0)
    print("\n" + report)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }

In [ ]:
args = TrainingArguments(
    "bert-punctuation-correction",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=logging_steps,
    fp16=True, # Speeds up training on GPU
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)


### Baseline

In [ ]:
model_baseline = get_baseline_model()

trainer = Trainer(
    model=model_baseline,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train() # get f1 score

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-sentence and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.141800,0.163423,0.946747,0.950601,0.947789,0.950601



              precision    recall  f1-score   support

           "       0.80      0.63      0.71      1102
           !       0.00      0.00      0.00        17
         ! -       0.00      0.00      0.00         1
          "        0.75      0.51      0.61       489
          ""       0.00      0.00      0.00         6
          ",       0.62      0.48      0.54       199
        ", -       0.00      0.00      0.00         6
          ".       0.70      0.61      0.65       355
          "?       0.00      0.00      0.00         1
           ,       0.82      0.83      0.83      8281
         , "       0.00      0.00      0.00        21
         , -       0.00      0.00      0.00         9
           -       0.71      0.44      0.55       913
         - "       0.00      0.00      0.00        24
           .       0.91      0.95      0.93      6632
         . "       0.00      0.00      0.00        22
         . -       0.64      0.17      0.27        52
           :       0.48   

TrainOutput(global_step=2386, training_loss=0.23794588652710302, metrics={'train_runtime': 546.7377, 'train_samples_per_second': 34.911, 'train_steps_per_second': 4.364, 'total_flos': 4162723170974688.0, 'train_loss': 0.23794588652710302, 'epoch': 1.0})

In [ ]:
save_path = "/content/drive/MyDrive/omg_diploma_2025/baseline_model"
trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/omg_diploma_2025/baseline_model/tokenizer_config.json',
 '/content/drive/MyDrive/omg_diploma_2025/baseline_model/special_tokens_map.json',
 '/content/drive/MyDrive/omg_diploma_2025/baseline_model/vocab.txt',
 '/content/drive/MyDrive/omg_diploma_2025/baseline_model/added_tokens.json',
 '/content/drive/MyDrive/omg_diploma_2025/baseline_model/tokenizer.json')

### LSTM run

In [ ]:
model_lstm = BertLSTM(MODEL_NAME, num_labels=NUM_LABELS)

trainer = Trainer(
    model=model_lstm,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
# Compare F1 with Baseline

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.180600,0.195120,0.936786,0.944842,0.939761,0.944842



              precision    recall  f1-score   support

           "       0.73      0.62      0.67      1102
           !       0.00      0.00      0.00        17
         ! -       0.00      0.00      0.00         1
          "        0.64      0.26      0.37       489
          ""       0.00      0.00      0.00         6
          ",       0.71      0.25      0.37       199
        ", -       0.00      0.00      0.00         6
          ".       0.71      0.49      0.58       355
          "?       0.00      0.00      0.00         1
           ,       0.81      0.81      0.81      8281
         , "       0.00      0.00      0.00        21
         , -       0.00      0.00      0.00         9
           -       0.61      0.35      0.45       913
         - "       0.00      0.00      0.00        24
           .       0.89      0.95      0.92      6632
         . "       0.00      0.00      0.00        22
         . -       0.00      0.00      0.00        52
           :       0.00   

TrainOutput(global_step=2386, training_loss=0.32194964442798696, metrics={'train_runtime': 579.871, 'train_samples_per_second': 32.916, 'train_steps_per_second': 4.115, 'total_flos': 0.0, 'train_loss': 0.32194964442798696, 'epoch': 1.0})

In [ ]:
save_path_lstm = "/content/drive/MyDrive/omg_diploma_2025/lstm_model"
os.makedirs(save_path_lstm, exist_ok=True)

torch.save(trainer.model.state_dict(), f"{save_path_lstm}/lstm_weights.pth")

tokenizer.save_pretrained(save_path_lstm)

trainer.model.config.save_pretrained(save_path_lstm)


# Evaluation & testing

In [ ]:
# # load the saved baseline model
# path = "/content/drive/MyDrive/omg_diploma_2025/baseline_model"

# model = AutoModelForTokenClassification.from_pretrained(path)
# tokenizer = AutoTokenizer.from_pretrained(path)

In [ ]:
# # load the saved lstm model

# class BertLSTM(nn.Module):
#     def __init__(self, model_name, num_labels):
#         super().__init__()
#         self.bert = AutoModel.from_pretrained(model_name)
#         self.config = self.bert.config

#         # LSTM Layer
#         self.lstm = nn.LSTM(input_size=768, hidden_size=256, batch_first=True, bidirectional=True)

#         # Classifier
#         self.classifier = nn.Linear(256 * 2, num_labels) # Bidirectional
#         self.num_labels = num_labels
#         self.loss_fct = nn.CrossEntropyLoss()

#     def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
#         outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
#         sequence_output = outputs.last_hidden_state

#         lstm_output, _ = self.lstm(sequence_output)

#         # Linear Layer
#         logits = self.classifier(lstm_output)

#         loss = None
#         if labels is not None:
#             loss = self.loss_fct(logits.view(-1, self.num_labels), labels.view(-1)) # Flatten to calculate loss

#         return (loss, logits) if loss is not None else logits


# path = "/content/drive/MyDrive/omg_diploma_2025/lstm_model"
# tokenizer = AutoTokenizer.from_pretrained(path)
# model = BertLSTM("DeepPavlov/rubert-base-cased-sentence", num_labels=NUM_LABELS)


# model.load_state_dict(torch.load(f"{path}/lstm_weights.pth", map_location="cpu"))
# model.eval() # Set to eval mode

In [ ]:
def evaluate_on_domain(domain_name):
    print(f"\n--- EVALUATING ON {domain_name.upper()} ---")
    filename = f"bench_test_{domain_name}.json"
    test_file_path = os.path.join(data_dir, filename)

    test_dataset = load_dataset("json", data_files={"test": test_file_path})

    if DEBUG:
        total_rows = len(test_dataset["test"])
        take_n = min(10, total_rows)

        print(f"Debug: Slicing {domain_name} test set (taking {take_n} from {total_rows})...")
        test_dataset["test"] = test_dataset["test"].select(range(take_n))

    tokenized_test = test_dataset.map(align_labels_with_tokens, batched=True)

    metrics = trainer.evaluate(tokenized_test["test"])

    print(f"Results for {domain_name}:")
    print(f"F1: {metrics['eval_f1']:.4f}")
    print(f"Precision: {metrics['eval_precision']:.4f}")
    print(f"Recall: {metrics['eval_recall']:.4f}")
    return metrics

In [ ]:
results_fiction = evaluate_on_domain("fiction")
results_journalism = evaluate_on_domain("journalism")
results_science = evaluate_on_domain("science")



--- EVALUATING ON FICTION ---


Generating test split: 0 examples [00:00, ? examples/s]

Debug: Slicing fiction test set (taking 3 from 3)...


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Restoration demo

In [ ]:
def reconstruct_text(tokens, labels, id2label):
    """
    Helper to stitch tokens and punctuation back into a string.
    """
    text = ""
    for token, label_id in zip(tokens, labels):
        # Skip special tokens if they appear
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        punct = id2label[label_id]

        # If label is 'O', it means just a space.
        # If it's punctuation, we attach it to the word.
        if punct == "O":
            text += token + " "
        else:
            # e.g. token="word", punct="," -> "word, "
            text += token + punct + " "

    return text.strip()


In [ ]:
def visualize_prediction(dataset, index=None):
    # 1. Pick a random example if no index
    if index is None:
        index = np.random.randint(0, len(dataset))

    sample = dataset[index]
    tokens = sample['tokens']
    true_label_ids = sample['ner_tags']

    device = next(trainer.model.parameters()).device

    tokenized_input = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    ).to(device)

    # 3. Predict
    with torch.no_grad():
        outputs = trainer.model(**tokenized_input)

        if isinstance(outputs, tuple):
            logits = outputs[1]
        elif hasattr(outputs, 'logits'):
            logits = outputs.logits
        else:
            logits = outputs
            if isinstance(logits, tuple):
                 logits = logits[1]

    predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

    word_ids = tokenized_input.word_ids()

    aligned_preds = []
    aligned_true = []
    aligned_tokens = []

    previous_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            continue

        if word_idx >= len(tokens):
            break

        aligned_tokens.append(tokens[word_idx])
        aligned_preds.append(id2label[predictions[i]])

        if word_idx < len(true_label_ids):
            try:
                 lbl_id = true_label_ids[word_idx]
                 if lbl_id in id2label:
                     aligned_true.append(id2label[lbl_id])
                 else:
                     aligned_true.append("O")
            except:
                 aligned_true.append("ERROR")
        else:
            aligned_true.append("UNKNOWN")

        previous_word_idx = word_idx

    df = pd.DataFrame({
        'Token': aligned_tokens,
        'Original Punct': aligned_true,
        'Predicted Punct': aligned_preds
    })

    df['Match'] = df['Original Punct'] == df['Predicted Punct']

    try:
        orig_ids = [label2id.get(l, 0) for l in aligned_true]
        pred_ids = [label2id.get(l, 0) for l in aligned_preds]

        original_text = reconstruct_text(aligned_tokens, orig_ids, id2label)
        predicted_text = reconstruct_text(aligned_tokens, pred_ids, id2label)

        print(f"--- Sample #{index} ---")
        print("\nORIGINAL TEXT:\n", original_text)
        print("\nPREDICTED TEXT:\n", predicted_text)
    except Exception as e:
        print(f"Error reconstructing text: {e}")

    print("\n--- Detailed Token Map (First 20 words) ---")
    display(df.head(50))

    return df


In [ ]:
# Takes a random sample from the 'test' part of our loaded dataset
_ = visualize_prediction(raw_datasets['validation'])


In [ ]:
def restore_punctuation(text):
    device = next(trainer.model.parameters()).device

    # Tokenize to split the text into words/subwords
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        is_split_into_words=False # tokenizer splits the string itself
    ).to(device)

    # Predict
    with torch.no_grad():
        outputs = trainer.model(**inputs)
        # Handle tuple return (loss, logits) vs just logits
        if isinstance(outputs, tuple):
            logits = outputs[1]
        elif hasattr(outputs, 'logits'):
            logits = outputs.logits
        else:
            logits = outputs

    # Get the ID of the most likely label
    predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

    # 4. Reconstruct
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    word_ids = inputs.word_ids()

    restored_text = ""
    previous_word_idx = None

    # Buffer to hold parts of a word (e.g. "под", "##сол", "##неч", "##ный")
    current_word_tokens = []
    current_punct = ""

    for i, word_idx in enumerate(word_ids):
        # Skip special tokens
        if word_idx is None:
            continue

        token = tokens[i]

        # If we moved to a NEW word, flush the previous one
        if word_idx != previous_word_idx:
            if previous_word_idx is not None:
                # Add the previous word to result
                full_word = "".join(current_word_tokens).replace("##", "")

                # Logic: If label is 'O', add space. If punct, add punct + space
                if current_punct == "O":
                    restored_text += full_word + " "
                else:
                    restored_text += full_word + current_punct + " "

            # Start new word
            current_word_tokens = [token]

            # Get the predicted punctuation for this NEW word (from its first token)
            pred_id = predictions[i]
            current_punct = id2label[pred_id]

        else:
            # We are still inside the same word (sub-tokens)
            current_word_tokens.append(token)

        previous_word_idx = word_idx

    # Flush the very last word
    if current_word_tokens:
        full_word = "".join(current_word_tokens).replace("##", "")
        if current_punct == "O":
            restored_text += full_word
        else:
            restored_text += full_word + current_punct

    return restored_text.strip()


In [ ]:
input_text = "мама мыла раму а папа читал газету"
print(f"Input:    {input_text}")
print(f"Restored: {restore_punctuation(input_text)}\n")

input_text_2 = "однако мы решили не идти в кино потому что пошел дождь"
print(f"Input:    {input_text_2}")
print(f"Restored: {restore_punctuation(input_text_2)}\n")

input_text_3 = "он сказал привет как дела"
print(f"Input:    {input_text_3}")
print(f"Restored: {restore_punctuation(input_text_3)}\n")
